<a href="https://colab.research.google.com/github/ahchu2525/capstone-hosp/blob/main/clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hotel Booking Cancellations - Data Preparation

(Prepared by Kelvin Chu)

**Problem statement:** Hotels lose revenue when guests cancel their bookings. Hotel managers need to know when and why guests cancel and which new bookings are likely to cancel.

In [3]:
import pandas as pd
import numpy as np

bookings = pd.read_csv('https://raw.githubusercontent.com/ahchu2525/capstone-hosp/main/Bookings.csv')
hotels = pd.read_csv('https://raw.githubusercontent.com/ahchu2525/capstone-hosp/main/Hotels.csv')
segments = pd.read_csv('https://raw.githubusercontent.com/ahchu2525/capstone-hosp/main/Market%20Segments.csv')
customer_types = pd.read_csv('https://raw.githubusercontent.com/ahchu2525/capstone-hosp/main/Customer%20Types.csv')
status = pd.read_csv('https://raw.githubusercontent.com/ahchu2525/capstone-hosp/main/Reservation%20Status.csv')

In [4]:
print('bookings:', bookings.shape)
print('hotels:', hotels.shape)
print('segments:', segments.shape)
print('customer_types:', customer_types.shape)
print('status:', status.shape)

bookings: (119390, 28)
hotels: (2, 2)
segments: (26, 3)
customer_types: (36, 4)
status: (3, 2)


## 2. Understand the data
Check each table for its structure, missing values, duplicates and odd values. All problems found are listed in the data issues log at the end of this section.

In [6]:
hotels

,Hotel_ID,hotel
0,1,Resort Hotel
1,2,City Hotel


In [7]:
status

,Reservation_ID,reservation_status
0,1,Check-Out
1,2,Canceled
2,3,No-Show


In [8]:
segments

,Market_Segment_ID,market_segment,distribution_channel
0,1,Direct,Direct
1,2,Corporate,Corporate
2,3,Online TA,TA/TO
3,4,Offline TA/TO,TA/TO
4,5,Complementary,Direct
5,6,Direct,TA/TO
6,7,Groups,Direct
7,8,Groups,Corporate
8,9,Offline TA/TO,Direct
9,10,Groups,TA/TO


In [9]:
customer_types

,Customer_Type_ID,customer_type,deposit_type,meal
0,1,Transient,No Deposit,BB
1,2,Transient,No Deposit,FB
2,3,Transient,No Deposit,HB
3,4,Contract,No Deposit,BB
4,5,Contract,No Deposit,HB
5,6,Transient-Party,No Deposit,BB
6,7,Transient-Party,No Deposit,FB
7,8,Transient-Party,No Deposit,HB
8,9,Group,No Deposit,BB
9,10,Transient-Party,Refundable,BB


**Observations - lookup tables**
- **hotels** has 2 hotels: Resort Hotel and City Hotel.
- **status** has 3 outcomes: Check-Out, Canceled and No-Show.
- **segments** has 26 rows. Each row is 1 pair of market segment and distribution channel. Some rows are labelled `Undefined`.
- **customer_types** has 36 rows. Each row is 1 mix of customer type, deposit type and meal plan. The meal plan has both `SC` and `Undefined`. Both mean no meal package.

In [10]:
bookings.head()

,Booking_ID,Hotel_ID,Market_Segment_ID,Customer_Type_ID,Reservation_ID,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,...,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,reservation_status_date
0,1,1,1,1,1,0,342,2015,July,27,...,0,0,3,NaN,NaN,0,0.0,0,0,2015-07-01
1,2,1,1,1,1,0,737,2015,July,27,...,0,0,4,NaN,NaN,0,0.0,0,0,2015-07-01
2,3,1,1,1,1,0,7,2015,July,27,...,0,0,0,NaN,NaN,0,75.0,0,0,2015-07-02
3,4,1,2,1,1,0,13,2015,July,27,...,0,0,0,304.0,NaN,0,75.0,0,0,2015-07-02
4,5,1,3,1,1,0,14,2015,July,27,...,0,0,0,240.0,NaN,0,98.0,0,1,2015-07-03


In [11]:
bookings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 28 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   Booking_ID                      119390 non-null  int64  
 1   Hotel_ID                        119390 non-null  int64  
 2   Market_Segment_ID               119390 non-null  int64  
 3   Customer_Type_ID                119390 non-null  int64  
 4   Reservation_ID                  119390 non-null  int64  
 5   is_canceled                     119390 non-null  int64  
 6   lead_time                       119390 non-null  int64  
 7   arrival_date_year               119390 non-null  int64  
 8   arrival_date_month              119390 non-null  object 
 9   arrival_date_week_number        119390 non-null  int64  
 10  arrival_date_day_of_month       119390 non-null  int64  
 11  stays_in_weekend_nights         119390 non-null  int64  
 12  stays_in_week_ni

In [12]:
bookings.describe()

,Booking_ID,Hotel_ID,Market_Segment_ID,Customer_Type_ID,Reservation_ID,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,...,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
count,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,...,119390.000000,119390.000000,119390.000000,119390.000000,103050.000000,6797.000000,119390.000000,119390.000000,119390.000000,119390.000000
mean,59695.500000,1.664461,4.262032,5.089044,1.380526,0.370416,104.011416,2016.156554,27.165173,15.798241,...,0.031912,0.087118,0.137097,0.221124,86.693382,189.266735,2.321149,101.831122,0.062518,0.571363
std,34465.068657,0.472181,3.071532,5.250009,0.505913,0.482918,106.863097,0.707476,13.605138,8.780829,...,0.175767,0.844336,1.497437,0.652306,110.774548,131.655015,17.594721,50.535790,0.245291,0.792798
min,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,2015.000000,1.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,1.000000,6.000000,0.000000,-6.380000,0.000000,0.000000
25%,29848.250000,1.000000,3.000000,1.000000,1.000000,0.000000,18.000000,2016.000000,16.000000,8.000000,...,0.000000,0.000000,0.000000,0.000000,9.000000,62.000000,0.000000,69.290000,0.000000,0.000000
50%,59695.500000,2.000000,3.000000,3.000000,1.000000,0.000000,69.000000,2016.000000,28.000000,16.000000,...,0.000000,0.000000,0.000000,0.000000,14.000000,179.000000,0.000000,94.575000,0.000000,0.000000
75%,89542.750000,2.000000,4.000000,8.000000,2.000000,1.000000,160.000000,2017.000000,38.000000,23.000000,...,0.000000,0.000000,0.000000,0.000000,229.000000,270.000000,0.000000,126.000000,0.000000,1.000000
max,119390.000000,2.000000,26.000000,36.000000,3.000000,1.000000,737.000000,2017.000000,53.000000,31.000000,...,1.000000,26.000000,72.000000,21.000000,535.000000,543.000000,391.000000,5400.000000,8.000000,5.000000


In [13]:
bookings.isnull().sum()

,0
Booking_ID,0
Hotel_ID,0
Market_Segment_ID,0
Customer_Type_ID,0
Reservation_ID,0
is_canceled,0
lead_time,0
arrival_date_year,0
arrival_date_month,0
arrival_date_week_number,0


In [14]:
print('Duplicate rows:', bookings.duplicated().sum())
print('Duplicate rows ignoring Booking_ID:', bookings.drop(columns='Booking_ID').duplicated().sum())

Duplicate rows: 0
Duplicate rows ignoring Booking_ID: 33010


In [15]:
print('Bookings with no guests:', ((bookings['adults'] + bookings['children'] + bookings['babies']) == 0).sum())
print('Bookings with 0 nights:', ((bookings['stays_in_weekend_nights'] + bookings['stays_in_week_nights']) == 0).sum())
print('ADR below 0:', (bookings['adr'] < 0).sum())
print('ADR equal to 0:', (bookings['adr'] == 0).sum())
print('Top 3 ADR values:', bookings['adr'].sort_values(ascending=False).head(3).tolist())

Bookings with no guests: 180
Bookings with 0 nights: 715
ADR below 0: 1
ADR equal to 0: 1959
Top 3 ADR values: [5400.0, 510.0, 508.0]


**Observations - missing values, duplicates and odd values**
- 4 columns have blanks. `company` is 94% blank so it is too empty to use.
- Every booking has its own ID. However 33,010 rows match another booking in every other column. These are likely several rooms in 1 group booking.
- 180 bookings have no guests and 1 has a negative ADR. These are not valid bookings.
- The top ADR of 5,400 is more than 10 times the next highest (510). It is likely a typing error.
- 1,959 bookings have an ADR of 0. Many are likely free stays so they are kept.

In [16]:
bookings['is_canceled'].value_counts()

,count
is_canceled,
0,75166
1,44224


In [17]:
bookings['arrival_date_year'].value_counts().sort_index()

,count
arrival_date_year,
2015,21996
2016,56707
2017,40687


In [18]:
bookings['country'].value_counts().head(10)

,count
country,
PRT,48590
GBR,12129
FRA,10415
ESP,8568
DEU,7287
ITA,3766
IRL,3375
BEL,2342
BRA,2224


In [19]:
print(bookings['Hotel_ID'].isin(hotels['Hotel_ID']).all())
print(bookings['Market_Segment_ID'].isin(segments['Market_Segment_ID']).all())
print(bookings['Customer_Type_ID'].isin(customer_types['Customer_Type_ID']).all())
print(bookings['Reservation_ID'].isin(status['Reservation_ID']).all())

True
True
True
True


### Data issues summary

| No. | Issue | Rows | Plan |
| --- | --- | --- | --- |
| 1 | `children` is blank | 4 | Fill with 0 |
| 2 | `country` is blank | 488 | Fill with `Unknown` |
| 3 | `agent` is blank | 16,340 | Means no agent. Fill with 0 and add a `has_agent` column |
| 4 | `company` is blank | 112,593 | Too empty to use. Add a `has_company` column then drop `company` |
| 5 | No adults, children or babies | 180 | Remove. A booking with no guests is not valid |
| 6 | `adr` below 0 | 1 | Remove |
| 7 | `adr` of 5,400 (next highest is 510) | 1 | Remove as a typing error |
| 8 | `adr` of 0 | 1,959 | Keep. Many are free stays |
| 9 | 0 nights booked | 715 | Keep. These add 0 revenue |
| 10 | Meal plan has both `SC` and `Undefined` | Count after join | Merge into `No meal` |
| 11 | Segment or channel is `Undefined` | Count after join | Remove. Too few to analyse |
| 12 | Arrival date split over 3 columns with month as text | All | Build 1 `arrival_date` column |
| 13 | Rows identical except `Booking_ID` | 33,010 | Keep for the dashboard. Likely several rooms in 1 group booking. Remove before modelling |
| 14 | `reservation_status` gives away the answer to `is_canceled` | All | Use in the dashboard only. Never use in the model |